::: {.callout-note}
## Wie man dieses Notebook lokal ausführt

1. Erstelle einen Ordner
2. Erstelle eine virtuelle Python-Umgebung  
   `python -m venv .venv`
3. Aktiviere die virtuelle Umgebung  
   `.venv\Scripts\activate`  
4. Installiere Jupyter und Pandas in dieser Umgebung     
   `pip install jupyter pandas ipython-sql sqlalchemy`
5. Lade diese Datei in deinen neu erstellten Ordner herunter
6. Starte den lokalen Jupyter-Server  
   `jupyter-lab`
:::

## Einleitung

In diesem Notebook sollen Anhand einer einfachen Datenbank die
Grunldagen von SQL repetiert werden. Damit SQL in einem Jupyter Notebook
verwendet werden und eine Datenbanktabelle dargestellt werden kann,
verwendet dieses Notebook die Python Libraries `sqlite3` und `pandas`.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

# Erstelle die zentrale Engine für den Arbeitsspeicher
engine = create_engine('sqlite:///:memory:')

## Inhalt der Datenbank

Die zum Training verwendete Datenbank bildet eine in einem Kurssystem
organisierte Schule ab. Kurssystem bedeutet dabei, dass Schülerinnen
und Schüler nicht einer fixen Klasse zugeteilt sind, sondern einzelne
Kurse besuchen. Die Beziehungen der daraus resultierenden Relationen
findet sich im ER Diagramm von @fig-erd.

![ER Diagramm der Datenbank der im Kurssystem organisierten
Schule.](kurssystem_idef1x_erd.svg){#fig-erd}

::: {.callout-note}
## Technischer Hinweis zur Verwendung von SQL in Jupyter Notebooks

Damit Übungen zu Datenbanken den Computer nicht mit Datenmüll belasten,
wird eine sogenannte 'In-Memory-Datenbank' angelegt. Verbindungen zu
realen Datenbanken werden mit SQLAlchemy über eine Engine erstellt, zum
Beispiel mit dem Befehl `engine =
create_engine('sqlite:///datenbank.db')`. Der String
`'sqlite:///datenbank.db'` fungiert dabei als Datenbank-URL
(Connection-String), die sowohl den Datenbank-Typ als auch den Pfad zur
Datei angibt.  Damit für die Übung eine Datenbank im
Arbeitsspeicher des Computers angelegt wird, muss die Adresse
`':memory:'` verwendet werden. Diese legt die Datenbank im Arbeitsspeicher
an. Entsprechend lautet der Befehl für das Erstellen einer Verbinung zu
einer Datenbank im Arbeitsspeicher `engine =
create_engine('sqlite:///:memory:')`. 

Eine weitere Besonderheit dieser Übungsdatenbank besteht darin, dass die
SQL-Befehle nicht direkt an die Datenbank gesendet werden. Stattdessen
laufen sie über die Python-Bibliothek SQLAlchemy. Dazu wird ein SQL-Befehl
als Text einer Python-Variabel zugewiesen. Der Text wird mit der
SQLAlchemy-Funktion `text()` als ausführbares SQLAlchemy-Objekt
gekennzeichnet und mit dem Befehl `pd.read_sql(variabel, engine)` an die
Datenbank übergeben und ausgeführt.

```Python
# 1. SQL Query in einer Python Variabel speichern
query = """
SELECT name
FROM Student
WHERE date_of_birth < '2003-12-31'
"""

# 2. Die Anfrage übergeben und das Resultat einem 
#    pandas Dataframe zuweisen
df_student = pd.read_sql(text(query), engine)
```
:::



In der folgenden Zelle wird die Definition der Datenbank der Schule gemäss obiger
Darstellung (vgl. @fig-erd) als SQL-Skript einer Variabel zugewiesen.

In [ ]:
# Setup - Inhalt nur verändern, wenn Sie wissen, was Sie tun


sql_ddl_script = """
-- Tabellen in der richtigen Reihenfolge erstellen (wegen Foreign Keys)

-- Erst löschen, falls vorhanden (Reihenfolge beachten!)
DROP TABLE IF EXISTS Mark;
DROP TABLE IF EXISTS Enrollment;
DROP TABLE IF EXISTS Student;
DROP TABLE IF EXISTS Class;
DROP TABLE IF EXISTS Course;

-- Dann neu erstellen

CREATE TABLE Course (
    course_id   INT PRIMARY KEY,
    subject     VARCHAR(100) NOT NULL,
    teacher     VARCHAR(100) NOT NULL
);

CREATE TABLE Class (
    class_id    INT PRIMARY KEY,
    course_id   INT NOT NULL,
    FOREIGN KEY (course_id) REFERENCES Course(course_id)
);

CREATE TABLE Student (
    student_no      INT PRIMARY KEY,
    name            VARCHAR(100) NOT NULL,
    prename         VARCHAR(100) NOT NULL,
    date_of_birth   DATE NOT NULL
);

CREATE TABLE Enrollment (
    student_no  INT NOT NULL,
    class_id    INT NOT NULL,
    PRIMARY KEY (student_no, class_id),
    FOREIGN KEY (student_no) REFERENCES Student(student_no),
    FOREIGN KEY (class_id)   REFERENCES Class(class_id)
);

CREATE TABLE Mark (
    mark_id     INT PRIMARY KEY,
    student_no  INT NOT NULL,
    class_id    INT NOT NULL,
    mark        DECIMAL(2,1) NOT NULL,
    datum       DATE NOT NULL,
    FOREIGN KEY (student_no, class_id) 
        REFERENCES Enrollment(student_no, class_id)
);
"""

Die nächste Code Zelle beinhaltet die Zuweisung der Daten Definition an
eine Variabel.

In [ ]:
sql_data = """
-- 1. Kurse
INSERT INTO Course (course_id, subject, teacher) VALUES 
(1, 'Deutsch', 'Frau Hürlimann'),
(2, 'Mathematik', 'Herr Vollenweider');

-- 2. Klassen
INSERT INTO Class (class_id, course_id) VALUES 
(201, 1), 
(202, 2);

-- 3. 20 Schüler
INSERT INTO Student (student_no, name, prename, date_of_birth) VALUES 
(1, 'Hugentobler', 'Urs', '2005-05-12'), 
(2, 'Zbinden', 'Beat', '2006-03-20'),
(3, 'Vogt', 'Flavia', '2005-10-02'), 
(4, 'Widmer', 'Reto', '2004-07-15'),
(5, 'Iten', 'Lukas', '2005-01-30'), 
(6, 'Moser', 'Sandra', '2006-11-14'),
(7, 'Gerber', 'Pascal', '2005-06-22'), 
(8, 'Brunner', 'Manuela', '2004-09-05'),
(9, 'Frei', 'Adrian', '2005-08-18'), 
(10, 'Lüthi', 'Fabienne', '2006-02-10'),
(11, 'Bachmann', 'Sven', '2005-04-25'), 
(12, 'Sutter', 'Corinne', '2004-12-12'),
(13, 'Keller', 'Marcel', '2005-02-28'), 
(14, 'Egger', 'Priska', '2006-07-03'),
(15, 'Graf', 'Silvan', '2005-03-17'), 
(16, 'Steiner', 'Jeannine', '2004-01-09'),
(17, 'Huber', 'Pirmin', '2005-11-01'), 
(18, 'Meier', 'Ursula', '2006-01-22'),
(19, 'Schmid', 'Remo', '2005-05-14'), 
(20, 'Bieri', 'Selina', '2004-10-30');

-- 4. Einschreibungen (gemischte Belegung)
INSERT INTO Enrollment (student_no, class_id) VALUES 
(1, 201), (1, 202), (2, 201), (2, 202), (3, 201), 
(3, 202), (4, 201), (4, 202), (5, 201), (5, 202),
(6, 201), (7, 201), (8, 201), (9, 201), (10, 201),
(11, 202), (12, 202), (13, 202), (14, 202), (15, 202),
(16, 201), (17, 201), (18, 202), (19, 202), (20, 201);

-- 5. Noten nach Schweizer Notenskala
INSERT INTO Mark (mark_id, student_no, class_id, mark, datum) VALUES 
(1, 1, 201, 5.5, '2025-01-15'), (2, 1, 202, 6.0, '2025-01-20'),
(3, 2, 201, 4.0, '2025-01-15'), (4, 2, 202, 5.0, '2025-01-20'),
(5, 3, 201, 3.5, '2025-01-15'), (6, 3, 202, 4.5, '2025-01-20'),
(7, 4, 201, 6.0, '2025-01-15'), (8, 4, 202, 5.5, '2025-01-20'),
(9, 5, 201, 4.5, '2025-01-15'), (10, 5, 202, 4.0, '2025-01-20'),
(11, 6, 201, 5.0, '2025-01-15'), (12, 11, 202, 3.0, '2025-01-20'),
(13, 12, 202, 4.5, '2025-01-20'), (14, 13, 202, 5.5, '2025-01-20'),
(15, 16, 201, 2.5, '2025-01-15'), (16, 17, 201, 4.0, '2025-01-15'),
(17, 18, 202, 5.0, '2025-01-20'), (18, 19, 202, 6.0, '2025-01-20'),
(19, 20, 201, 4.5, '2025-01-15'), (20, 10, 201, 5.5, '2025-01-15');
"""

Weil die SQL-Skripte für die Definition der Datenbank lang sind und
SQLite über SQLAlchemy nur einen Befehl pro Aufruf erlaubt, werden die
Skripte am Semikolon aufgetrennt. Die einzelnen Befehle werden dann in
einer Schleife nacheinander ausgeführt.

In [ ]:
# Die DDL- und Daten-Skripte bleiben gleich 
# (deine Variablen sql_ddl_script und sql_data)

with engine.begin() as conn:
    # Wir führen die Skripte über die Engine aus
    # Hinweis: SQLite erlaubt via SQLAlchemy meist 
    # nur einen Befehl pro Aufruf,
    # daher splitten wir das Skript am Semikolon.
    for statement in sql_ddl_script.split(';'):
        if statement.strip():
            conn.execute(text(statement))
            
    for statement in sql_data.split(';'):
        if statement.strip():
            conn.execute(text(statement))

print("Datenbank erfolgreich im RAM erstellt und befüllt!")

Datenbank erfolgreich im RAM erstellt und befüllt!


Nun stehen die Daten in den jeweiligen Tabellen zur Verfügung. Um das zu
überprüfen, wird in der folgenden Code-Zelle die Tabelle mit den Angaben
zu den Schülerinnen und Schülern einem pandas Dataframe zugewiesen und
dieser dann angezeigt.

In [5]:
sql_sus = """
SELECT *
FROM Student
"""

df_student = pd.read_sql(sql_sus, engine)

print(df_student.to_string(index=False))

 student_no        name  prename date_of_birth
          1 Hugentobler      Urs    2005-05-12
          2     Zbinden     Beat    2006-03-20
          3        Vogt   Flavia    2005-10-02
          4      Widmer     Reto    2004-07-15
          5        Iten    Lukas    2005-01-30
          6       Moser   Sandra    2006-11-14
          7      Gerber   Pascal    2005-06-22
          8     Brunner  Manuela    2004-09-05
          9        Frei   Adrian    2005-08-18
         10       Lüthi Fabienne    2006-02-10
         11    Bachmann     Sven    2005-04-25
         12      Sutter  Corinne    2004-12-12
         13      Keller   Marcel    2005-02-28
         14       Egger   Priska    2006-07-03
         15        Graf   Silvan    2005-03-17
         16     Steiner Jeannine    2004-01-09
         17       Huber   Pirmin    2005-11-01
         18       Meier   Ursula    2006-01-22
         19      Schmid     Remo    2005-05-14
         20       Bieri   Selina    2004-10-30


## Anwendungen

### Filtern mit `WHERE`

Die folgende Abfrage soll alle Schülerinnen und Schüler (Name und Vorname)
ausgegeben, welche am 1. April 2023 noch keine 18 Jahre alt waren.  

In [8]:
sql_kleiner_18 = """
SELECT name, prename
FROM Student
WHERE date_of_birth > '2005-03-31'
"""

df_kleiner_18 = pd.read_sql(text(sql_kleiner_18), engine)

print(df_kleiner_18.to_string(index=False))


       name  prename
Hugentobler      Urs
    Zbinden     Beat
       Vogt   Flavia
      Moser   Sandra
     Gerber   Pascal
       Frei   Adrian
      Lüthi Fabienne
   Bachmann     Sven
      Egger   Priska
      Huber   Pirmin
      Meier   Ursula
     Schmid     Remo
